In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# plt.rcParams['font.family'] = 'Malgun Gothic' # For Windows
plt.rcParams["font.family"] = "AppleGothic"   # Mac
%matplotlib inline

In [2]:
import pandas as pd

df = pd.read_csv("./2025_Airbnb_NYC_listings.csv")
df.head()

,Unnamed: 0,id,source,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,0,36121,city scrape,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,Full of tree-lined streets and beautiful brown...,62165,Michael,2009-12-11,"New York, NY",...,5.00,5.00,5.00,NaN,f,1,0,1,0,0.05
1,1,36647,city scrape,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,"Manhattan, SE corner of 2nd Ave/ E. 110th street",157798,Irene,2010-07-04,"New York, NY",...,4.90,4.38,4.71,NaN,f,1,0,1,0,0.58
2,2,38663,city scrape,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...","diverse, lively, hip, cool: loaded with restau...",165789,Sarah,2010-07-13,"New York, NY",...,4.88,4.86,4.62,OSE-STRREG-0001784,f,1,0,1,0,0.28
3,3,38833,city scrape,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,West Harlem is now packed with great restauran...,166532,Matthew,2010-07-14,"New York, NY",...,4.96,4.79,4.82,OSE-STRREG-0000476,f,1,1,0,0,1.36
4,4,39282,city scrape,“Work-from-home” from OUR home.,*Monthly Discount will automatically apply <br...,THE NEIGHBORHOOD:<br />Our apartment is locate...,168525,Gustavo,2010-07-16,"New York, NY",...,4.88,4.85,4.78,OSE-STRREG-0001150,f,2,0,2,0,1.54


In [3]:
df.shape

(22308, 73)

# 단기 (합법) / 중기(30박이상) / 호텔 (class b 존재여부) 나누기
- 숙소변수
 property_type
 room_type
- 합법 변수
 license
- 30 박 이상
minimum_nights
- class B 뭐야.

In [43]:
df[['property_type']]

,property_type
0,Private room in rental unit
1,Private room in condo
2,Private room in home
3,Entire home
4,Private room in rental unit
...,...
22303,Private room in rental unit
22304,Private room in rental unit
22305,Entire rental unit
22306,Entire rental unit


In [6]:
df['property_type'].value_counts()

property_type
Entire rental unit                    9648
Private room in rental unit           5189
Private room in home                  1886
Entire home                           1059
Room in hotel                          814
Entire condo                           691
Private room in townhouse              622
Private room in condo                  341
Entire townhouse                       336
Entire loft                            291
Entire guest suite                     272
Room in boutique hotel                 212
Entire serviced apartment              145
Private room in guest suite            114
Private room in loft                    62
Entire place                            61
Private room in bed and breakfast       60
Private room in serviced apartment      60
Shared room in rental unit              59
Room in aparthotel                      57
Private room in casa particular         41
Entire guesthouse                       38
Private room                            

- Room in aparthotel -> 호텔 /class B로 분류
-Entire rental unit                    9648 ->


In [7]:
df['room_type'].value_counts()

room_type
Entire home/apt    12664
Private room        9186
Hotel room           372
Shared room           86
Name: count, dtype: int64

# property 과 room 컬럼 의미/ 분석 목적
-property
-숙박 공간의 정체성(공간의 법적 정체성)
-entire 주거용/숙박용(호텔과같은) 나뉘어져있음
* roomtype만 쓰면 주거용 entire과 숙박용 entire 으로 나눌수 없어 규제 대상이 섞인다.
-따라, 주거/숙박업으로 나누어  법규제 적용 여부를 본다. -> 운영이 가능한 트랙을 결정

-room type
-게스트들이 어떻게 쓰이는 공간인지
- 가격차이, 수요 성향 -> 운영방식, 가격전략
-단독으로 쓰면 규제를 받는 건지 마는건지 명확하지 않음.

핵심 : 두 컬럼 같이본다. 


In [8]:
df[['property_type','room_type']].head(10)

,property_type,room_type
0,Private room in rental unit,Private room
1,Private room in condo,Private room
2,Private room in home,Private room
3,Entire home,Entire home/apt
4,Private room in rental unit,Private room
5,Private room in rental unit,Private room
6,Entire rental unit,Entire home/apt
7,Private room in condo,Private room
8,Private room in rental unit,Private room
9,Private room in guest suite,Private room


In [9]:
len(df['property_type'])

22308

In [10]:
df['property_type'].value_counts().head(12)

property_type
Entire rental unit             9648
Private room in rental unit    5189
Private room in home           1886
Entire home                    1059
Room in hotel                   814
Entire condo                    691
Private room in townhouse       622
Private room in condo           341
Entire townhouse                336
Entire loft                     291
Entire guest suite              272
Room in boutique hotel          212
Name: count, dtype: int64

In [11]:
# 표본수 200개 이상만 뽑아 본다.
top12_property_types = (df['property_type'].value_counts().head(12).index.tolist())
top12_property_types

['Entire rental unit',
 'Private room in rental unit',
 'Private room in home',
 'Entire home',
 'Room in hotel',
 'Entire condo',
 'Private room in townhouse',
 'Private room in condo',
 'Entire townhouse',
 'Entire loft',
 'Entire guest suite',
 'Room in boutique hotel']

# vlaue_counts로 본 property_type의 종류를 거주용/숙박용 기준으로 나누어서 새 컬럼에 매핑.
- Residential   (주거형)
- Lodging       (숙박업형)

In [12]:
property_map = {
    'Room in hotel': 'Lodging',
    'Room in boutique hotel': 'Lodging',

    'Entire rental unit': 'Residential',
    'Private room in rental unit': 'Residential',
    'Private room in home': 'Residential',
    'Entire home': 'Residential',
    'Entire condo': 'Residential',
    'Private room in townhouse': 'Residential',
    'Private room in condo': 'Residential',
    'Entire townhouse': 'Residential',
    'Entire loft': 'Residential',
    'Entire guest suite': 'Residential'
}

In [13]:
df['property_regulation_type'] = df['property_type'].map(property_map)

In [36]:
df['property_regulation_type'].value_counts()

property_regulation_type
Residential    20335
Lodging         1026
Other            947
Name: count, dtype: int64

In [15]:
df['property_regulation_type'].isna().value_counts()

property_regulation_type
False    21361
True       947
Name: count, dtype: int64

In [16]:
df['property_regulation_type'] = df['property_regulation_type'].fillna('Other')

In [17]:
len(df['property_regulation_type'])

22308

- Residential      ->  entire hom/apt         -> 단기 30박미만 => 불법 
-                                             -> 단기 30박 이상  => 합법
-                 -> private room/shared room -=> exempt/합법
- Lodging       ->  단기 합법 가능 

In [ ]:
df['legal_flag'] = 'Legal'  # 합법 컬럼 생성해주기.

# Residential + Entire + 단기(<30박) → 불법 가능
df.loc[
    (df['property_regulation_type'] == 'Residential') &
    (df['room_type'] == 'Entire home/apt') &
    (df['minimum_nights'] < 30),
    'legal_flag'
] = 'Illegal'

# 나머지는 Legal 그대로

In [27]:
df['legal_flag'].value_counts()

legal_flag
Legal      20808
Illegal     1500
Name: count, dtype: int64

In [33]:
# 최종 분석 데이터셋 확정하기
df_final = df[
    ( (df['property_regulation_type'] == 'Residential') & (df['legal_flag'] == 'Legal') )
    | ( df['property_regulation_type'] == 'Lodging' )
]

In [ ]:
# 불법숙소제거 약 11% - 안정수준
df_final.shape

(19861, 75)

In [34]:
df_final

,Unnamed: 0,id,source,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,...,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,property_regulation_type,legal_flag
0,0,36121,city scrape,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,Full of tree-lined streets and beautiful brown...,62165,Michael,2009-12-11,"New York, NY",...,5.00,NaN,f,1,0,1,0,0.05,Residential,Legal
1,1,36647,city scrape,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,"Manhattan, SE corner of 2nd Ave/ E. 110th street",157798,Irene,2010-07-04,"New York, NY",...,4.71,NaN,f,1,0,1,0,0.58,Residential,Legal
2,2,38663,city scrape,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...","diverse, lively, hip, cool: loaded with restau...",165789,Sarah,2010-07-13,"New York, NY",...,4.62,OSE-STRREG-0001784,f,1,0,1,0,0.28,Residential,Legal
4,4,39282,city scrape,“Work-from-home” from OUR home.,*Monthly Discount will automatically apply <br...,THE NEIGHBORHOOD:<br />Our apartment is locate...,168525,Gustavo,2010-07-16,"New York, NY",...,4.78,OSE-STRREG-0001150,f,2,0,2,0,1.54,Residential,Legal
5,5,39572,city scrape,1 br in a 2 br apt (Midtown West),NaN,NaN,169927,Hubert,2010-07-17,"Saint-Aubin-sur-Scie, France",...,4.86,NaN,f,2,1,1,0,0.25,Residential,Legal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22303,37429,1366333532374850165,city scrape,Beautiful 1-Bed Apt in Harlem!,Welcome to your temporary home in the vibrant ...,NaN,40019013,Cecilia,2015-07-30,"New York, NY",...,NaN,NaN,f,1,0,1,0,NaN,Residential,Legal
22304,37430,1366717321390111215,city scrape,Private Room w/ Ensuite Bath H,Stylish Private Rooms w/ En-Suite Baths in Bro...,NaN,483056418,Kristina,2022-10-10,"New York, NY",...,NaN,NaN,f,24,0,24,0,NaN,Residential,Legal
22305,37431,1366721904709517353,city scrape,2 Bedroom on East Side,Located in the Murray Hill area and occupies a...,NaN,30283594,Global Luxury Suites,2015-03-30,"Hawthorne, NJ",...,NaN,NaN,f,48,48,0,0,NaN,Residential,Legal
22306,37432,1366722692755341871,city scrape,Stylish 2Bd near Bryant Park,Enjoy everything the city has to offer while l...,NaN,407304997,Boomerang,2021-06-15,"New York, NY",...,NaN,NaN,t,25,25,0,0,NaN,Residential,Legal


여기까지. 주거용/숙소용을 나눴다.
다음 할 것
- 주거용 중에서 합법과불법인 숙소 뽑기 
'property_regulation_type' * room_type

- entire home/apr가   residential 이면 불법가능
- 


다음 할 것
- 단기/ 중기/ 호텔로 나누어 뽑기

내일 모델링 (가격)
숙소 나누고, 
가격


In [68]:
s = df.loc[df['property_regulation_type'] == 'Residential', ['property_regulation_type', 'room_type']]
s

,property_regulation_type,room_type
0,Residential,Private room
1,Residential,Private room
2,Residential,Private room
3,Residential,Entire home/apt
4,Residential,Private room
...,...,...
22303,Residential,Private room
22304,Residential,Private room
22305,Residential,Entire home/apt
22306,Residential,Entire home/apt


### license 합 불법 나누기

In [56]:
df['license'].value_counts()

license
Exempt                2064
OSE-STRREG-0000068      81
OSE-STRREG-0041458      24
OSE-STRREG-0001054      17
OSE-STRREG-0001957       9
                      ... 
OSE-STRREG-0002916       1
OSE-STRREG-0002238       1
OSE-STRREG-0002922       1
OSE-STRREG-0002813       1
OSE-STRREG-0002894       1
Name: count, Length: 1894, dtype: int64

- 0 - 불법
- 1 - 등록번호 있음
- 2- Exempt
---------------
- illegal (불법)
- registered (등록된)
- Exempt (면제된,적용대상에서 제외)

In [58]:
def license_status(x):
    if pd.isna(x):
        return 0      # illegal
    elif x == 'Exempt':
        return 2      # exempt
    else:
        return 1      # registered

df['license_status'] = df['license'].apply(license_status)

In [60]:
df['license_status'].value_counts()

license_status
0    17845
1     2399
2     2064
Name: count, dtype: int64